# Notebook 2 · The Simplex Method in Action

**Algorithms & Geometry of Linear Programming — IBS Summer School, August 2026**

Companion notebook to Lecture 2, *The Simplex Method and Diameter of Polyhedra*. Estimated time: **30 minutes**.

| Part | Topic | Lecture section | Time |
|---|---|---|---|
| A | Step through the walk, pivot by pivot | The simplex method | ≈ 10 min |
| B | Pivot rules, compared | Pivot rules | ≈ 10 min |
| C | The exponential worst case: the Klee–Minty cube | Pivot rules / diameter | ≈ 10 min |

**How to work — no programming is required.** Run the setup cell, then proceed from top to bottom. You will interact in three ways only: **move sliders and dropdowns**; **edit plainly marked numbers** (a single objective vector, in Part A); and **answer checkpoints** by replacing the `...` after an `=` sign with a number or a short word, then running the check cell below it. All algorithms — the simplex method, all pivot rules, and the Klee–Minty generator — are implemented for you inside the collapsed setup cell.

Throughout, LPs are in the computational form of the lecture, $\max\{c^\top x : Ax \le b,\ x \ge 0\}$ with $b \ge 0$, so that the origin is a vertex and the slack basis is feasible (this sidesteps the two-phase initialization discussed in the lecture).


In [ ]:
#@title Setup — the simplex method, pivot rules, Klee–Minty, and all drawing (run me) { display-mode: "form" }
# ---------------------------------------------------------------------------
# Requirements: numpy, scipy, matplotlib, plotly, ipywidgets, nbformat.
# In Google Colab all of these are preinstalled and nothing needs to be done.
# When running locally (VS Code, plain Jupyter), uncomment the line below and
# run this cell once to install them into the active kernel.
# ---------------------------------------------------------------------------
# %pip install numpy scipy matplotlib plotly ipywidgets nbformat
# (after installing, restart the kernel so the packages are picked up)
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations
from collections import deque
from scipy.optimize import linprog

np.set_printoptions(precision=4, suppress=True)


def solve_lp(A, b, c):
    A = np.asarray(A, float); b = np.asarray(b, float); c = np.asarray(c, float)
    res = linprog(-c, A_ub=A, b_ub=b,
                  bounds=[(None, None)] * A.shape[1], method="highs")
    if res.status == 0:
        return "optimal", res.x, float(c @ res.x)
    return {2: "infeasible", 3: "unbounded"}.get(res.status, "error"), None, None


def tight_set(A, b, x, tol=1e-6):
    A = np.asarray(A, float); b = np.asarray(b, float)
    return [i for i in range(len(b))
            if abs(A[i] @ x - b[i]) <= tol * (1 + abs(b[i]))]


def enumerate_vertices(A, b, tol=1e-7):
    A = np.asarray(A, float); b = np.asarray(b, float)
    m, n = A.shape
    cand = []
    for I in combinations(range(m), n):
        A_I = A[list(I)]
        if np.linalg.matrix_rank(A_I) < n:
            continue
        x = np.linalg.solve(A_I, b[list(I)])
        if np.all(A @ x <= b + tol * (1 + np.abs(b))):
            cand.append(x)
    if not cand:
        return np.zeros((0, n))
    return np.unique(np.round(np.array(cand), 8), axis=0)


def polytope_edges(A, b, V, tol=1e-6):
    A = np.asarray(A, float); n = A.shape[1]
    tights = [set(tight_set(A, b, v, tol)) for v in V]
    E = []
    for p in range(len(V)):
        for q in range(p + 1, len(V)):
            common = sorted(tights[p] & tights[q])
            if len(common) >= n - 1 and np.linalg.matrix_rank(A[common]) == n - 1:
                E.append((p, q))
    return E


# ---------------------------------------------------------------- the simplex
def simplex(A, b, c, pivot_rule, max_iter=100000, seed=0, tol=1e-9):
    """max c^T x s.t. A x <= b, x >= 0 (b >= 0), starting at the vertex 0.
    Returns status, x, value, pivots, path (visited vertices), log (one line
    per pivot).  Variables 0..n-1 are x1..xn; n..n+m-1 are the slacks s1..sm."""
    A = np.asarray(A, float); b = np.asarray(b, float); c = np.asarray(c, float)
    m, n = A.shape
    assert np.all(b >= -1e-12), "this implementation requires b >= 0"
    rng = np.random.default_rng(seed)
    T = np.hstack([A, np.eye(m), b.reshape(-1, 1)])
    z = np.concatenate([c, np.zeros(m)])
    value, basis = 0.0, list(range(n, n + m))

    def vertex():
        x = np.zeros(n)
        for i, j in enumerate(basis):
            if j < n:
                x[j] = T[i, -1]
        return x

    def name(j):
        return f"x{j+1}" if j < n else f"s{j-n+1}"

    path, log = [vertex()], []
    for pivots in range(max_iter):
        candidates = [j for j in range(n + m) if z[j] > tol]
        if not candidates:
            return dict(status="optimal", x=vertex(), value=value,
                        pivots=pivots, path=np.array(path), log=log)
        j = pivot_rule(candidates, z, rng)
        rows = [i for i in range(m) if T[i, j] > tol]
        if not rows:
            return dict(status="unbounded", x=None, value=np.inf,
                        pivots=pivots, path=np.array(path), log=log)
        r = min(rows, key=lambda i: (T[i, -1] / T[i, j], basis[i]))
        entering, leaving = name(j), name(basis[r])
        T[r] /= T[r, j]
        for i in range(m):
            if i != r:
                T[i] -= T[i, j] * T[r]
        value += z[j] * T[r, -1]
        z = z - z[j] * T[r, :-1]
        basis[r] = j
        path.append(vertex())
        log.append(f"pivot {pivots+1:3d}: {entering:>3} enters, {leaving:>3} "
                   f"leaves   ->  vertex {np.round(vertex(), 3)},  "
                   f"objective {value:.4f}")
    return dict(status="max_iter", x=vertex(), value=value, pivots=max_iter,
                path=np.array(path), log=log)


def dantzig_rule(candidates, z, rng):
    """Dantzig: enter the candidate with the largest reduced cost."""
    return max(candidates, key=lambda j: z[j])


def bland_rule(candidates, z, rng):
    """Bland: enter the candidate with the smallest index (anti-cycling)."""
    return min(candidates)


def random_edge_rule(candidates, z, rng):
    """Random edge: enter a uniformly random candidate."""
    return int(rng.choice(candidates))


RULES = {"Dantzig": dantzig_rule, "Bland": bland_rule,
         "random edge": random_edge_rule}


RULE_NAMES = ("Dantzig", "steepest edge", "shadow vertex")


def simplex_rule(A, b, c, rule, max_iter=100000, tol=1e-9):
    """The simplex method on max{c^T x : A x <= b, x >= 0} (b >= 0), started
    at the vertex 0, under one of three pivot rules:

      "Dantzig"       -- enter the variable with the largest reduced cost;
      "steepest edge" -- enter the variable maximizing the reduced cost per
                         unit length of the corresponding edge direction;
      "shadow vertex" -- the parametric objective rule: with u equal to -1 on
                         the initial nonbasic variables (so that the starting
                         vertex is its unique maximizer), follow the vertices
                         optimal for (1 - lam) u + lam c as lam grows to 1.

    Returns a dict with status, x, value, pivots, path."""
    A = np.asarray(A, float); b = np.asarray(b, float); c = np.asarray(c, float)
    m, n = A.shape
    assert np.all(b >= -1e-12), "this implementation requires b >= 0"
    T = np.hstack([A, np.eye(m), b.reshape(-1, 1)])
    z = np.concatenate([c, np.zeros(m)])
    zu = np.concatenate([-np.ones(n), np.zeros(m)])   # auxiliary objective u
    basis = list(range(n, n + m))
    in_basis = [False] * n + [True] * m

    def vertex():
        x = np.zeros(n)
        for i, j in enumerate(basis):
            if j < n:
                x[j] = T[i, -1]
        return x

    path = [vertex()]
    for pivots in range(max_iter):
        if rule == "shadow vertex":
            # entering variable: the reduced cost (1 - lam) zu_j + lam z_j
            # that turns positive first as lam grows
            j, lam_j = None, np.inf
            for q in range(n + m):
                if in_basis[q]:
                    continue
                den = z[q] - zu[q]
                if den > tol and -zu[q] / den < lam_j - 1e-12:
                    lam_j, j = -zu[q] / den, q
            if j is None or lam_j > 1.0 + 1e-9:
                x = vertex()
                return dict(status="optimal", x=x, value=float(c @ x),
                            pivots=pivots, path=np.array(path))
        else:
            cand = [q for q in range(n + m) if z[q] > tol]
            if not cand:
                x = vertex()
                return dict(status="optimal", x=x, value=float(c @ x),
                            pivots=pivots, path=np.array(path))
            if rule == "Dantzig":
                j = max(cand, key=lambda q: z[q])
            elif rule == "steepest edge":
                j = max(cand, key=lambda q: z[q] / np.sqrt(1.0 + T[:, q] @ T[:, q]))
            else:
                raise ValueError(f"unknown rule {rule!r}")
        rows = [i for i in range(m) if T[i, j] > tol]
        if not rows:
            return dict(status="unbounded", x=None, value=np.inf,
                        pivots=pivots, path=np.array(path))
        r = min(rows, key=lambda i: (T[i, -1] / T[i, j], basis[i]))
        T[r] /= T[r, j]
        for i in range(m):
            if i != r:
                T[i] -= T[i, j] * T[r]
        z = z - z[j] * T[r, :-1]
        zu = zu - zu[j] * T[r, :-1]
        in_basis[basis[r]] = False
        in_basis[j] = True
        basis[r] = j
        path.append(vertex())
    return dict(status="max_iter", x=None, value=None, pivots=max_iter,
                path=np.array(path))


def pivot_growth_data(ns, trials=20, seed=0):
    """Mean pivot counts of the three rules on random LPs with n variables
    and m = 2n random constraints (plus x >= 0), for each n in ns."""
    out = {rule: [] for rule in RULE_NAMES}
    for n in ns:
        for rule in RULE_NAMES:
            cts = [simplex_rule(*random_lp(n, 2 * n, seed=seed + 1000 * n + s),
                                rule)["pivots"] for s in range(trials)]
            out[rule].append(float(np.mean(cts)))
    return out


def klee_minty(n):
    """The Klee--Minty LP in n variables (Matousek--Gartner formulation)."""
    A = np.zeros((n, n)); b = np.zeros(n)
    for i in range(n):
        for j in range(i):
            A[i, j] = 2 * 3.0**(i - j)
        A[i, i] = 1.0
        b[i] = 9.0**i
    c = np.array([3.0**(n - 1 - j) for j in range(n)])
    return A, b, c


def random_lp(n=6, m=20, seed=0):
    """A random bounded LP with feasible origin and c > 0."""
    rng = np.random.default_rng(seed)
    A = np.vstack([rng.standard_normal((m, n)), np.eye(n)])
    b = np.concatenate([rng.uniform(1.0, 2.0, size=m), np.full(n, 10.0)])
    return A, b, rng.uniform(0.5, 1.5, size=n)


# ------------------------------------------------------------------- drawing
def _display_vertices(A, b, box, tol=1e-7):
    A = np.asarray(A, float); b = np.asarray(b, float)
    Bx = np.array([[1., 0.], [-1., 0.], [0., 1.], [0., -1.]])
    dx = np.array([box[1], -box[0], box[3], -box[2]], float)
    A2, b2 = np.vstack([A, Bx]), np.concatenate([b, dx])
    V = []
    for i, j in combinations(range(len(b2)), 2):
        M = A2[[i, j]]
        if abs(np.linalg.det(M)) < tol:
            continue
        v = np.linalg.solve(M, b2[[i, j]])
        if np.all(A2 @ v <= b2 + 1e-6):
            V.append(v)
    if not V:
        return np.zeros((0, 2))
    V = np.unique(np.round(np.array(V), 8), axis=0)
    ctr = V.mean(axis=0)
    ang = np.arctan2(V[:, 1] - ctr[1], V[:, 0] - ctr[0])
    return V[np.argsort(ang)]


def draw_lp(A, b, c=None, labels=None, xlim=(-1, 5), ylim=(-1, 4), ax=None,
            show_normals=True):
    A = np.asarray(A, float); b = np.asarray(b, float)
    if ax is None:
        _, ax = plt.subplots(figsize=(6.2, 6.2))
    span = max(xlim[1] - xlim[0], ylim[1] - ylim[0])
    L = 2.0 * span
    ctr = np.array([(xlim[0] + xlim[1]) / 2, (ylim[0] + ylim[1]) / 2])
    V = _display_vertices(A, b, (xlim[0], xlim[1], ylim[0], ylim[1]))
    if len(V) >= 3:
        ax.fill(V[:, 0], V[:, 1], color="#9ecae1", alpha=0.55, zorder=1)
    for i, (a, bi) in enumerate(zip(A, b)):
        na = np.linalg.norm(a)
        if na < 1e-12:
            continue
        p0 = a * bi / na**2
        d = np.array([-a[1], a[0]]) / na
        seg = np.array([p0 - L * d, p0 + L * d])
        lab = labels[i] if labels is not None and i < len(labels) else None
        (h,) = ax.plot(seg[:, 0], seg[:, 1], lw=1.5, label=lab, zorder=2)
        if show_normals:
            foot = ctr - (a / na) * ((a / na) @ ctr - bi / na)
            tip = foot - 0.06 * span * a / na
            ax.annotate("", xy=tip, xytext=foot, zorder=2,
                        arrowprops=dict(arrowstyle="->", color=h.get_color()))
    if c is not None:
        c = np.asarray(c, float)
        status, xopt, val = solve_lp(A, b, c)
        if status == "optimal":
            nc = np.linalg.norm(c)
            delta = 0.15 * span * nc
            for k in range(3):
                t = val - k * delta
                p0 = c * t / nc**2
                d = np.array([-c[1], c[0]]) / nc
                seg = np.array([p0 - L * d, p0 + L * d])
                ax.plot(seg[:, 0], seg[:, 1], "--", color="gray",
                        lw=2.0 if k == 0 else 0.9, zorder=3)
            ax.plot(*xopt, "o", ms=9, mfc="gold", mec="black", zorder=6)
    if labels is not None:
        ax.legend(loc="upper right", fontsize=8, framealpha=0.9)
    ax.set_xlim(xlim); ax.set_ylim(ylim); ax.set_aspect("equal")
    ax.axhline(0, color="black", lw=0.5, zorder=0)
    ax.axvline(0, color="black", lw=0.5, zorder=0)
    return ax


def simplex_scrubber(A, b, c, labels=None, xlim=(-1, 5), ylim=(-1, 4),
                     rule=None, title=""):
    """Run the simplex method once, then replay it with a slider over the
    pivot number: the figure shows the walk so far, the text the log line."""
    A = np.asarray(A, float); b = np.asarray(b, float); c = np.asarray(c, float)
    rule = dantzig_rule if rule is None else rule
    res = simplex(A, b, c, pivot_rule=rule)
    P, log, K = res["path"], res["log"], res["pivots"]

    def show(pivot=0):
        _, ax = plt.subplots(figsize=(6.2, 6.2))
        A_draw = np.vstack([A, -np.eye(2)])
        b_draw = np.concatenate([b, [0.0, 0.0]])
        draw_lp(A_draw, b_draw, c=c, labels=labels, xlim=xlim, ylim=ylim, ax=ax)
        for k in range(pivot):
            ax.annotate("", xy=P[k + 1], xytext=P[k], zorder=7,
                        arrowprops=dict(arrowstyle="-|>", lw=2.2,
                                        color="darkred"))
        for k in range(pivot + 1):
            ax.plot(*P[k], "o", ms=6, color="darkred", zorder=8)
            ax.annotate(str(k), P[k], textcoords="offset points",
                        xytext=(-12, 5), color="darkred", fontsize=10, zorder=8)
        ax.plot(*P[pivot], "o", ms=11, mfc="none", mec="darkred", mew=2,
                zorder=9)
        ax.set_title(f"{title}   —   pivot {pivot} of {K}")
        plt.show()
        if pivot == 0:
            print(f"start: vertex {np.round(P[0], 3)},  "
                  f"objective {float(c @ P[0]):.4f}")
        else:
            print(log[pivot - 1])
        if pivot == K:
            print(f"OPTIMAL: no improving edge remains at "
                  f"x* = {np.round(res['x'], 3)}, value {res['value']:.4f}.")

    try:
        from ipywidgets import interact, IntSlider
        interact(show, pivot=IntSlider(value=0, min=0, max=K, step=1,
                                       continuous_update=False))
    except Exception:
        for k in range(K + 1):
            show(k)


def km3d_scrubber():
    """Replay Dantzig's walk on KM(3) on the wireframe of the deformed cube.
    Coordinates are rescaled by (1, 9, 81) for display only."""
    A, b, c = klee_minty(3)
    res = simplex(A, b, c, pivot_rule=dantzig_rule)
    P, log, K = res["path"], res["log"], res["pivots"]
    A_full = np.vstack([A, -np.eye(3)])
    b_full = np.concatenate([b, np.zeros(3)])
    V = enumerate_vertices(A_full, b_full)
    E = polytope_edges(A_full, b_full, V)
    s = np.array([1.0, 9.0, 81.0])
    Vd, Pd = V / s, P / s

    def show(pivot=0):
        fig = plt.figure(figsize=(7, 6))
        ax = fig.add_subplot(projection="3d")
        for (p, q) in E:
            ax.plot(*zip(Vd[p], Vd[q]), color="steelblue", lw=1.2, alpha=0.8)
        ax.plot(Pd[:pivot + 1, 0], Pd[:pivot + 1, 1], Pd[:pivot + 1, 2],
                color="darkred", lw=2.5, marker="o", ms=4)
        ax.scatter(*Pd[0], color="black", s=45)
        ax.text(*Pd[0], "  start", fontsize=10)
        ax.scatter(*Pd[pivot], color="darkred", s=60)
        ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$"); ax.set_zlabel("$x_3$")
        ax.set_title(f"Dantzig on KM(3) — pivot {pivot} of {K}\n"
                     "(coordinates rescaled for display)")
        ax.view_init(elev=18, azim=-63)
        plt.tight_layout(); plt.show()
        print(f"start: vertex {np.round(P[0], 3)}" if pivot == 0
              else log[pivot - 1])

    try:
        from ipywidgets import interact, IntSlider
        interact(show, pivot=IntSlider(value=0, min=0, max=K, step=1,
                                       continuous_update=False))
    except Exception:
        show(K)


def pivot_rule_lab(A, b, c, labels=None, xlim=(-1, 5), ylim=(-1, 4)):
    """Choose a pivot rule from the dropdown and see its walk on the polygon
    (the sign constraints x >= 0 are added for drawing)."""
    def show(rule="Dantzig"):
        r = simplex_rule(np.asarray(A, float), np.asarray(b, float),
                         np.asarray(c, float), rule)
        _, ax = plt.subplots(figsize=(5.6, 5.6))
        A_draw = np.vstack([A, -np.eye(2)])
        b_draw = np.concatenate([np.asarray(b, float), [0.0, 0.0]])
        draw_lp(A_draw, b_draw, c=c, labels=labels, xlim=xlim, ylim=ylim, ax=ax)
        P = r["path"]
        for k in range(len(P) - 1):
            ax.annotate("", xy=P[k + 1], xytext=P[k], zorder=7,
                        arrowprops=dict(arrowstyle="-|>", lw=2.2,
                                        color="darkred"))
        ax.set_title(f"{rule}: {r['pivots']} pivots on the polygon")
        plt.show()

    try:
        from ipywidgets import interact, Dropdown
        interact(show, rule=Dropdown(options=list(RULE_NAMES),
                                     value="Dantzig", description="pivot rule"))
    except Exception:
        for rule in RULE_NAMES:
            show(rule)


def graph_distance(A, b, x_start, x_end):
    """BFS distance between two vertices of {A x <= b, x >= 0} in the
    vertex-edge graph."""
    A = np.asarray(A, float); b = np.asarray(b, float)
    n = A.shape[1]
    A_full = np.vstack([A, -np.eye(n)])
    b_full = np.concatenate([b, np.zeros(n)])
    V = enumerate_vertices(A_full, b_full)
    E = polytope_edges(A_full, b_full, V)
    adj = {i: [] for i in range(len(V))}
    for p, q in E:
        adj[p].append(q); adj[q].append(p)

    def locate(x):
        d = np.linalg.norm(V - np.asarray(x, float), axis=1)
        return int(np.argmin(d / (1 + np.linalg.norm(V, axis=1))))

    s, t = locate(x_start), locate(x_end)
    dist = {s: 0}; Q = deque([s])
    while Q:
        u = Q.popleft()
        if u == t:
            return dist[u]
        for w in adj[u]:
            if w not in dist:
                dist[w] = dist[u] + 1
                Q.append(w)
    return None



# --------------------------------------------- 3D drawing (self-contained)

def _lp3_solve(A, b, c):
    from scipy.optimize import linprog as _linprog
    A = np.asarray(A, float); b = np.asarray(b, float); c = np.asarray(c, float)
    res = _linprog(-c, A_ub=A, b_ub=b, bounds=[(None, None)] * A.shape[1],
                   method="highs")
    if res.status == 0:
        return "optimal", res.x, float(c @ res.x)
    return {2: "infeasible", 3: "unbounded"}.get(res.status, "error"), None, None


def _lp3_tight(A, b, x, tol=1e-6):
    A = np.asarray(A, float); b = np.asarray(b, float)
    return [i for i in range(len(b))
            if abs(A[i] @ x - b[i]) <= tol * (1 + abs(b[i]))]


def _lp3_vertices(A, b, tol=1e-7):
    from itertools import combinations as _comb
    A = np.asarray(A, float); b = np.asarray(b, float)
    m, n = A.shape
    cand = []
    for I in _comb(range(m), n):
        A_I = A[list(I)]
        if np.linalg.matrix_rank(A_I) < n:
            continue
        x = np.linalg.solve(A_I, b[list(I)])
        if np.all(A @ x <= b + tol * (1 + np.abs(b))):
            cand.append(x)
    if not cand:
        return np.zeros((0, n))
    return np.unique(np.round(np.array(cand), 8), axis=0)


def _lp3_edges(A, b, V, tol=1e-6):
    from itertools import combinations as _comb
    A = np.asarray(A, float)
    E = []
    for i, j in _comb(range(len(V)), 2):
        Ti, Tj = _lp3_tight(A, b, V[i], tol), _lp3_tight(A, b, V[j], tol)
        common = [k for k in Ti if k in Tj]
        if len(common) >= 2 and np.linalg.matrix_rank(A[common]) == 2:
            E.append((i, j))
    return E


def _lp3_bounded(A, b):
    for e in np.eye(A.shape[1]):
        for sgn in (1.0, -1.0):
            if _lp3_solve(A, b, sgn * e)[0] == "unbounded":
                return False
    return True


def show_lp_3d(A, b, c=None, title=None, show_tight_sets=True,
               path=None, curve=None,
               path_label="step", curve_label="central path"):
    """Interactive 3D view of P = {x : A x <= b} in R^3 (drag to rotate).
    Hovering over a vertex shows its coordinates and (if show_tight_sets)
    its tight set I; the origin is marked in green.  With an objective c,
    the optimal vertex is highlighted in red and the objective drawn as an
    arrow based there.  A piecewise-linear walk can be overlaid via `path`
    (dark red, one marker per point) and a smooth curve via `curve`
    (orange)."""
    A = np.asarray(A, float); b = np.asarray(b, float)
    V = _lp3_vertices(A, b)
    status = None
    if c is not None:
        status, x_opt, val = _lp3_solve(A, b, c)
        if status == "infeasible":
            print("This system is INFEASIBLE: no point satisfies all constraints.")
            return
        if status == "unbounded":
            print("The LP is UNBOUNDED: the objective increases without limit "
                  "along a recession direction of the region.")
    if len(V) == 0:
        if _lp3_solve(A, b, np.zeros(A.shape[1]))[0] == "infeasible":
            print("This system is INFEASIBLE: no point satisfies all constraints.")
        else:
            print("The region is nonempty but has no vertices "
                  "(it is unbounded and contains a line).")
        return
    bounded = _lp3_bounded(A, b)
    if not bounded:
        print("Note: the region is UNBOUNDED — the figure shows only its "
              "vertices and bounded edges; the region extends beyond them.")
    E = _lp3_edges(A, b, V)

    hover = []
    for v in V:
        txt = f"x = ({v[0]:.2f}, {v[1]:.2f}, {v[2]:.2f})"
        if show_tight_sets:
            txt += f"<br>I = {_lp3_tight(A, b, v)}"
        hover.append(txt)

    head = title or ""
    if c is not None and status == "optimal":
        I_opt = _lp3_tight(A, b, x_opt)
        line = (f"x* = ({x_opt[0]:.2f}, {x_opt[1]:.2f}, {x_opt[2]:.2f}),  "
                f"value = {val:.2f},  tight set I = {I_opt}")
        head = f"{head}<br>{line}" if head else line

    try:
        import plotly.graph_objects as go
        data = []
        if bounded and len(V) >= 4:
            try:
                from scipy.spatial import ConvexHull
                hull = ConvexHull(V, qhull_options="QJ")
                data.append(go.Mesh3d(
                    x=V[:, 0], y=V[:, 1], z=V[:, 2],
                    i=hull.simplices[:, 0], j=hull.simplices[:, 1],
                    k=hull.simplices[:, 2],
                    color="#9ecae1", opacity=0.35, flatshading=True,
                    hoverinfo="skip"))
            except Exception:
                pass
        ex, ey, ez = [], [], []
        for i, j in E:
            ex += [V[i, 0], V[j, 0], None]
            ey += [V[i, 1], V[j, 1], None]
            ez += [V[i, 2], V[j, 2], None]
        data.append(go.Scatter3d(x=ex, y=ey, z=ez, mode="lines",
                                 line=dict(color="#404040", width=4),
                                 hoverinfo="skip"))
        data.append(go.Scatter3d(
            x=V[:, 0], y=V[:, 1], z=V[:, 2], mode="markers",
            marker=dict(size=5, color="black"),
            text=hover, hoverinfo="text"))
        # the origin, marked for reference
        data.append(go.Scatter3d(
            x=[0.0], y=[0.0], z=[0.0], mode="markers",
            marker=dict(size=3.5, color="green"),
            text=["origin (0, 0, 0)"], hoverinfo="text"))
        if curve is not None:
            Cv = np.asarray(curve, float)
            data.append(go.Scatter3d(
                x=Cv[:, 0], y=Cv[:, 1], z=Cv[:, 2], mode="lines",
                line=dict(color="darkorange", width=7),
                text=[curve_label] * len(Cv), hoverinfo="text"))
        if path is not None:
            Pt = np.asarray(path, float)
            data.append(go.Scatter3d(
                x=Pt[:, 0], y=Pt[:, 1], z=Pt[:, 2], mode="lines+markers",
                line=dict(color="darkred", width=6),
                marker=dict(size=4.5, color="darkred"),
                text=[f"{path_label} {k}" for k in range(len(Pt))],
                hoverinfo="text"))
            data.append(go.Scatter3d(
                x=[Pt[0, 0]], y=[Pt[0, 1]], z=[Pt[0, 2]], mode="markers",
                marker=dict(size=7, color="black"),
                text=["start"], hoverinfo="text"))
        if c is not None and status == "optimal":
            data.append(go.Scatter3d(
                x=[x_opt[0]], y=[x_opt[1]], z=[x_opt[2]], mode="markers",
                marker=dict(size=9, color="crimson", symbol="diamond"),
                text=["optimum"], hoverinfo="text"))
            cn = np.asarray(c, float)
            cn = cn / np.linalg.norm(cn)
            span = float(np.max(V.max(axis=0) - V.min(axis=0)))
            tip = x_opt + 0.35 * span * cn
            data.append(go.Scatter3d(
                x=[x_opt[0], tip[0]], y=[x_opt[1], tip[1]],
                z=[x_opt[2], tip[2]], mode="lines",
                line=dict(color="crimson", width=6), hoverinfo="skip"))
            data.append(go.Cone(
                x=[tip[0]], y=[tip[1]], z=[tip[2]],
                u=[cn[0]], v=[cn[1]], w=[cn[2]],
                sizemode="absolute", sizeref=0.12 * span,
                anchor="tail", showscale=False,
                colorscale=[[0, "crimson"], [1, "crimson"]],
                hoverinfo="skip"))
        fig = go.Figure(data=data)
        fig.update_layout(
            title=dict(text=head, font=dict(size=13)),
            scene=dict(xaxis_title="x1", yaxis_title="x2", zaxis_title="x3",
                       aspectmode="data",
                       # initial viewpoint from the (+x, -y, +z) octant, so
                       # that x1 increases from left to right on the screen
                       camera=dict(eye=dict(x=1.5, y=-1.5, z=1.0))),
            margin=dict(l=0, r=0, t=60, b=0), showlegend=False,
            width=650, height=550)
        fig.show()
    except ImportError:
        # Static fallback if plotly is unavailable.
        from mpl_toolkits.mplot3d.art3d import Line3DCollection
        fig = plt.figure(figsize=(6.5, 6.0))
        ax = fig.add_subplot(projection="3d")
        segs = [[V[i], V[j]] for i, j in E]
        ax.add_collection3d(Line3DCollection(segs, colors="#404040", lw=1.5))
        ax.scatter(V[:, 0], V[:, 1], V[:, 2], color="black", s=25)
        ax.scatter([0], [0], [0], color="green", s=15)
        if show_tight_sets:
            for v in V:
                ax.text(v[0], v[1], v[2], f" I={_lp3_tight(A, b, v)}", fontsize=7)
        if curve is not None:
            Cv = np.asarray(curve, float)
            ax.plot(Cv[:, 0], Cv[:, 1], Cv[:, 2], color="darkorange", lw=2.5)
        if path is not None:
            Pt = np.asarray(path, float)
            ax.plot(Pt[:, 0], Pt[:, 1], Pt[:, 2], color="darkred",
                    lw=2.0, marker="o", ms=4)
        if c is not None and status == "optimal":
            ax.scatter(*x_opt, color="crimson", s=80, zorder=5)
        ax.set_xlabel("x1"); ax.set_ylabel("x2"); ax.set_zlabel("x3")
        ax.set_title(head.replace("<br>", "\n"), fontsize=10)
        plt.show()

def tableau_demo(seed=0):
    """Replay the full run of Dantzig's rule on a random instance
    min{c^T x : A x <= b, x >= 0} with A in {1..5}^{4x3}, b in {1..5} and
    c in {-5..-1}, displaying the dictionary (tableau) after every pivot,
    in exact arithmetic.  The basis starts at (s_1, ..., s_4), i.e. x = 0,
    s = b.  The method stops when every reduced cost is nonnegative.
    Change the seed for a fresh instance."""
    from fractions import Fraction
    from IPython.display import display, Markdown, Math

    rng = np.random.default_rng(seed)
    m, n = 4, 3
    A = rng.integers(1, 6, size=(m, n))
    b = rng.integers(1, 6, size=m)
    c = -rng.integers(1, 6, size=n)          # c in {-5, ..., -1}^3
    names = [f"x_{j+1}" for j in range(n)] + [f"s_{i+1}" for i in range(m)]

    # Dictionary state: nonbasic list N, basic list B (variable indices);
    # rows[i] = (const, {j: coef}) meaning  var B[i] = const + sum coef_j var_j,
    # zrow = (value, {j: reduced cost}) meaning  z = value + sum rc_j var_j.
    N = list(range(n))
    B = list(range(n, n + m))
    rows = [(Fraction(int(b[i])), {j: Fraction(-int(A[i, j])) for j in N})
            for i in range(m)]
    zrow = (Fraction(0), {j: Fraction(int(c[j])) for j in N})

    def fmt_coef(f):
        if f.denominator == 1:
            return str(abs(f.numerator))
        return r"\tfrac{%d}{%d}" % (abs(f.numerator), f.denominator)

    def fmt_terms(coefs, mark=None):
        out = ""
        for j in N:
            f = coefs.get(j, Fraction(0))
            if f == 0:
                continue
            sign = "+" if f > 0 else "-"
            mag = fmt_coef(f)
            coef_txt = "" if mag == "1" else mag
            term = r"%s %s%s" % (sign, coef_txt, names[j])
            if j == mark:
                term = r"\textcolor{red}{%s}" % term
            out += r"\; " + term + r"\;"
        return out

    def fmt_const(f):
        if f.denominator == 1:
            return str(f.numerator)
        return r"%s\tfrac{%d}{%d}" % ("-" if f < 0 else "",
                                      abs(f.numerator), f.denominator)

    def show_dict(enter=None, leave_row=None):
        lines = [r"z &= %s %s" % (fmt_const(zrow[0]), fmt_terms(zrow[1], enter))]
        for i in range(m):
            name = names[B[i]]
            if i == leave_row:
                name = r"\textcolor{blue}{%s}" % name
            lines.append(r"%s &= %s %s"
                         % (name, fmt_const(rows[i][0]), fmt_terms(rows[i][1])))
        display(Math(r"\begin{aligned}" + r"\\[2pt] ".join(lines)
                     + r"\end{aligned}"))

    def solution():
        x = [Fraction(0)] * (n + m)
        for i in range(m):
            x[B[i]] = rows[i][0]
        return x

    display(Markdown(
        "**Instance** (seed = %d):  minimize $c^\\top x$ subject to "
        "$Ax \\le b$, $x \\ge 0$, with\n\n"
        "$A = %s$, &nbsp; $b = %s$, &nbsp; $c = %s$."
        % (seed,
           r"\begin{pmatrix}" + r"\\".join("&".join(str(int(A[i, j]))
                for j in range(n)) for i in range(m)) + r"\end{pmatrix}",
           r"(" + ", ".join(str(int(v)) for v in b) + r")^\top",
           r"(" + ", ".join(str(int(v)) for v in c) + r")^\top")))
    display(Markdown("**Starting dictionary** — basis $(s_1, \\dots, s_4)$, "
                     "current vertex $x = 0$, $s = b$, value $0$:"))

    for pivot in range(1, 51):
        # Dantzig: entering variable of most negative reduced cost
        cand = [j for j in N if zrow[1].get(j, Fraction(0)) < 0]
        if not cand:
            show_dict()
            x = solution()
            display(Markdown(
                "**Optimal.**  Every reduced cost in the $z$-row is "
                "nonnegative, so no entering variable remains: the current "
                "vertex $x^* = (%s)$ with value $z = %s$ is optimal."
                % (", ".join(str(x[j]) for j in range(n)), fmt_const(zrow[0]))))
            return
        e = min(cand, key=lambda j: (zrow[1][j], j))
        # ratio test among rows with negative coefficient of the entering var
        ratio_rows = [i for i in range(m)
                      if rows[i][1].get(e, Fraction(0)) < 0]
        r = min(ratio_rows, key=lambda i: (-rows[i][0] / rows[i][1][e], B[i]))
        ratio = -rows[r][0] / rows[r][1][e]
        show_dict(enter=e, leave_row=r)
        display(Markdown(
            "**Pivot %d.**  Entering: $\\textcolor{red}{%s}$ (most negative "
            "reduced cost $%s$).  Ratio test: $%s$ leaves first, at $%s = %s$ "
            "(row marked blue).  New dictionary:"
            % (pivot, names[e], fmt_const(zrow[1][e]), names[B[r]],
               names[e], fmt_const(ratio))))

        # pivot: solve row r for the entering variable, substitute everywhere
        C, coefs = rows[r]
        a = coefs[e]                       # coefficient of entering var (< 0)
        new_coefs = {B[r]: 1 / a}
        for j in N:
            if j != e and coefs.get(j, Fraction(0)) != 0:
                new_coefs[j] = -coefs[j] / a
        new_const = -C / a
        leaving = B[r]
        N[N.index(e)] = leaving
        B[r] = e
        rows[r] = (new_const, new_coefs)

        def substitute(const, coefs):
            f = coefs.pop(e, Fraction(0))
            if f == 0:
                return (const, {j: v for j, v in coefs.items() if v != 0})
            const = const + f * new_const
            out = dict(coefs)
            for j, v in new_coefs.items():
                out[j] = out.get(j, Fraction(0)) + f * v
            return (const, {j: v for j, v in out.items() if v != 0})

        for i in range(m):
            if i != r:
                rows[i] = substitute(*rows[i])
        zrow = substitute(*zrow)
    display(Markdown("*(stopped after 50 pivots — degenerate cycling?)*"))

print("Setup complete: simplex, RULES, klee_minty, and all scrubbers defined.")


## Part A — Watching the walk (≈ 10 min)

The simplex method turns the picture of Notebook 1 into an algorithm: start at a vertex, and as long as some incident edge improves the objective, move along it. By the correctness theorem of the lecture, a vertex with no improving incident edge is *globally* optimal, so the walk cannot terminate at a false optimum. Introducing slacks $s = b - Ax \ge 0$, a **basis** selects which variables are expressed in terms of the others, and a **pivot** exchanges one basic and one nonbasic variable — algebraically, one element of the tight set of Notebook 1; geometrically, one edge of the polyhedron.

The next cell replays the algorithm on the production LP of Lecture 1 with a slider over the pivot number. At each position, the figure shows the walk so far (dark-red arrows, current vertex circled) and the text reports the same pivot algebraically: which variable *enters* the basis, which *leaves*, the new vertex, and the objective value. Step through and verify that each arrow crosses exactly one edge and that the objective strictly increases.


In [ ]:
# The production LP (the sign constraints x >= 0 are implicit in this form,
# so A holds only the two resource rows).
A_prod = np.array([[1.0, 1.0],     # wood
                   [1.0, 3.0]])    # labor
b_prod = np.array([4.0, 6.0])
c_prod = np.array([2.0, 3.0])

simplex_scrubber(A_prod, b_prod, c_prod,
                 labels=["wood", "labor", "x1 >= 0", "x2 >= 0"],
                 xlim=(-0.5, 5), ylim=(-0.5, 4),
                 title="production LP, Dantzig's rule")


Below, the same replay on the polygon of Notebook 1. The objective is written out in plain sight: **you may edit the two numbers in `c_mg` and rerun the cell** to watch different walks over the same polygon.


In [ ]:
c_mg = np.array([1.0, 1.0])       # <- edit these two numbers and rerun

A_mg = np.array([[-1.0, 1.0],     # x2 - x1  <= 1
                 [ 1.0, 6.0],     # x1 + 6x2 <= 15
                 [ 4.0,-1.0]])    # 4x1 - x2 <= 10
b_mg = np.array([1.0, 15.0, 10.0])

simplex_scrubber(A_mg, b_mg, c_mg,
                 labels=["x2 - x1 <= 1", "x1 + 6x2 <= 15", "4x1 - x2 <= 10",
                         "x1 >= 0", "x2 >= 0"],
                 xlim=(-1, 5), ylim=(-1, 4),
                 title="Notebook 1 polygon, Dantzig's rule")


### Checkpoint A.1 — a walk that overshoots

Set `c_mg = np.array([1.0, 1.1])` in the cell above, rerun it, and step through the walk. Record below **how many pivots** the method takes, then run the check. Watch the route carefully: does the walk take the shortest path to the optimum?


In [ ]:
pivots_with_c_1_11 = ...     # A.1: pivots taken by Dantzig's rule for c = (1, 1.1)


In [ ]:
# --- Check for A.1 -----------------------------------------------------------
assert pivots_with_c_1_11 is not Ellipsis, "replace ... by a number first"
truth = simplex(A_mg, b_mg, np.array([1.0, 1.1]), pivot_rule=dantzig_rule)
assert pivots_with_c_1_11 == truth["pivots"], \
    (f"the run takes {truth['pivots']} pivots — step through the slider again")
print(f"A.1 correct: {truth['pivots']} pivots, along "
      "(0,0) -> (0,1) -> (1.29, 2.29) -> (3,2).")
print("The walk overshoots to (1.29, 2.29) and then doubles back along the")
print("boundary: the greedy local choice need not take the shortest route.")
print("The optimum is 2 edges from the start, yet the rule used 3.")


**The tableau view.** The geometric walk has exact algebraic bookkeeping: the *dictionary* (equivalently, the simplex tableau). Introduce a slack variable $s_i = b_i - a_i^\top x \ge 0$ for each constraint and write $z = c^\top x$ for the objective. Solving for the basic variables expresses $z$ and the basis in terms of the nonbasic variables, which are held at $0$: the constants on the right-hand sides then read off the current solution, the constant in the $z$-row is its objective value, and the coefficients of the nonbasic variables in the $z$-row are the *reduced costs*. Each pivot of Dantzig's rule, as stated in the lecture, enters the nonbasic variable of most negative reduced cost (marked red), removes the basic variable that the ratio test drops to $0$ first (marked blue), and re-solves the system for the new basis; since we are minimizing, the method stops when every reduced cost is nonnegative. The cell below replays a full run in exact arithmetic on a random instance $\min\{c^\top x : Ax \le b,\ x \ge 0\}$ with $A \in \{1, \dots, 5\}^{4 \times 3}$, $b$ with entries in $\{1, \dots, 5\}$, and $c$ with entries in $\{-5, \dots, -1\}$ — the constraint data are positive, so $x = 0$, $s = b$ is a feasible starting vertex and the LP is bounded. Change the seed and re-run for a fresh instance.

In [ ]:
tableau_demo(seed=0)   # <- change the seed and re-run for a fresh instance

## Part B — Pivot rules, compared (≈ 10 min)

At a typical vertex several edges improve the objective, and the **pivot rule** chooses among them; correctness does not depend on the choice (any improving edge makes progress), but the number of pivots does. Three rules are implemented in the setup:

- **Dantzig** — enter the variable with the *largest reduced cost*, the steepest immediate gain per unit of the entering variable;
- **steepest edge** — enter the variable whose edge direction maximizes the *gain per unit of distance traveled*; this refinement is the workhorse of modern implementations;
- **shadow vertex** — the *parametric objective* rule: take the auxiliary objective $u$ equal to $-1$ on the complement of the starting basis (so the starting vertex is its unique maximizer — equivalently, $u$ rewards minimizing the sum of the initially nonbasic variables), and follow the vertices that are optimal for the interpolated objectives $(1 - \lambda)\, u + \lambda\, c$ as $\lambda$ grows from $0$ to $1$. This is the rule for which the smoothed analysis of Lecture 3 is carried out.

### Checkpoint B.1 — predict before measuring

Which of the three rules will be **fastest on average** on random LPs? Record your prediction (write exactly one of `"Dantzig"`, `"steepest edge"`, `"shadow vertex"`), then explore the walks with the dropdown and run the measurement cell, which tests all three rules on instances of growing size.

In [ ]:
predicted_fastest_rule = "..."    # B.1: "Dantzig", "steepest edge", or "shadow vertex"

In [ ]:
pivot_rule_lab(A_mg, b_mg, np.array([1.0, 1.1]),
               labels=["x2 - x1 <= 1", "x1 + 6x2 <= 15", "4x1 - x2 <= 10",
                       "x1 >= 0", "x2 >= 0"])


In [ ]:
# The measurement: mean pivot counts of the three rules on random LPs of
# growing size (n variables, m = 2n random constraints, plus x >= 0),
# 20 instances per size.  Shamir's survey of computational experience
# [Sha87] reports that on typical instances the number of pivots grows
# only *linearly* in the dimensions of A — the dashed lines are the
# least-squares linear fits, one per rule.
ns_grow = list(range(2, 21, 2))
growth = pivot_growth_data(ns_grow, trials=20)

fig, ax = plt.subplots(figsize=(7.0, 4.6))
markers = {"Dantzig": "o", "steepest edge": "s", "shadow vertex": "^"}
rule_means = {}
for rule in RULE_NAMES:
    ys = growth[rule]
    rule_means[rule] = ys[-1]
    slope, icept = np.polyfit(ns_grow, ys, 1)
    (h,) = ax.plot(ns_grow, ys, marker=markers[rule], ms=5, lw=1.6,
                   label=f"{rule}  (fit {slope:.2f} n {icept:+.2f})")
    ax.plot(ns_grow, slope * np.array(ns_grow) + icept, "--", lw=1.0,
            color=h.get_color(), alpha=0.6)
ax.set_xlabel("n  (variables; m = 2n random constraints)")
ax.set_ylabel("mean pivots over 20 random LPs")
ax.set_title("pivot counts grow linearly in n on random LPs, for all three rules")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.show()

In [ ]:
# --- Check for B.1 -----------------------------------------------------------
assert predicted_fastest_rule != "...", "write one of the three rule names first"
fastest = min(rule_means, key=rule_means.get)
guess = str(predicted_fastest_rule).strip().lower()
assert guess == fastest.lower(), \
    (f"at the largest size the fastest was {fastest} "
     f"({rule_means[fastest]:.1f} pivots on average) — compare the plot")
print(f"B.1 correct: {fastest} was fastest on average "
      f"({rule_means[fastest]:.1f} pivots at n = {ns_grow[-1]}).")
print("All three rules track a straight line: the linear growth on typical")
print("instances reported in Shamir's survey.  Note the units: our A is")
print("m x n before slacks, so n is the number of nonbasic variables of the")
print("standard-form program — the quantity in which the growth is linear.")
print("Yet no pivot rule is known to be polynomial in the worst case;")
print("Part C constructs the classical exponential example.")

## Part C — The Klee–Minty cube (≈ 10 min)

The lecture's worst-case construction is a *deformed cube* that fools Dantzig's rule into visiting all $2^n$ vertices. In the Matoušek–Gärtner formulation (implemented for you as `klee_minty(n)`):
$$\max\ \sum_{j=1}^{n} 3^{\,n-j}\, x_j \quad\text{s.t.}\quad 2 \sum_{j=1}^{i-1} 3^{\,i-j}\, x_j + x_i \le 9^{\,i-1} \quad (i = 1, \dots, n), \qquad x \ge 0.$$
For $n = 3$ this is exactly the instance drawn in the lecture: $\max\, 9x_1 + 3x_2 + x_3$ subject to $x_1 \le 1$, $\ 6x_1 + x_2 \le 9$, $\ 18x_1 + 6x_2 + x_3 \le 81$, $\ x \ge 0$.

The next cell measures Dantzig's rule on KM($n$) for $n = 2, \dots, 11$ and plots the pivot counts on a logarithmic scale.


In [ ]:
ns = list(range(2, 12))
piv = [simplex(*klee_minty(n), pivot_rule=dantzig_rule)["pivots"] for n in ns]
for n, p in zip(ns, piv):
    print(f"n = {n:2d}:  {p:5d} pivots")

fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.semilogy(ns, piv, "o-", color="darkred", label="Dantzig pivots on KM($n$)")
ax.semilogy(ns, [2**n - 1 for n in ns], "--", color="gray", label="$2^n - 1$")
ax.set_xlabel("$n$"); ax.set_ylabel("pivots (log scale)")
ax.grid(alpha=0.3); ax.legend()
ax.set_title("Exponential worst case for Dantzig's rule")
plt.show()


### Checkpoint C.1 — extrapolate the pattern

Read the pattern off the table above and predict the number of pivots for $n = 12$. Record your prediction and run the check, which performs the (sizeable) run.


In [ ]:
predicted_pivots_n12 = ...     # C.1: pivots on KM(12) under Dantzig's rule


In [ ]:
# --- Check for C.1 -----------------------------------------------------------
assert predicted_pivots_n12 is not Ellipsis, "replace ... by a number first"
r12 = simplex(*klee_minty(12), pivot_rule=dantzig_rule)
assert predicted_pivots_n12 == r12["pivots"], \
    (f"KM(12) took {r12['pivots']} pivots — look at the pattern "
     "3, 7, 15, 31, ... in the table")
print(f"C.1 correct: {r12['pivots']} = 2^12 - 1 pivots.  The method visits")
print("every one of the 2^12 = 4096 vertices: one pivot per vertex after the")
print("first.  Ten more variables would take about a million pivots.")


The scrubber below replays the $n = 3$ walk on the wireframe of the deformed cube (coordinates rescaled by $1, 9, 81$ for display; the combinatorics is unchanged). Step through all seven pivots and watch the walk snake through every vertex — and note where the optimum sits relative to the start.


In [ ]:
km3d_scrubber()


**The whole walk at once.** The scrubber above replays the walk one pivot at a time on a static wireframe. The cell below draws the complete walk of Dantzig's rule on the deformed cube $KM(3)$ as a rotatable three-dimensional figure (coordinates rescaled by $(1, 9, 81)$ for display, as above): the red path starts at the origin and passes through all $2^3 = 8$ vertices before reaching the optimum. Rotate the figure and follow the path from face to face; hovering over a path marker shows its step number.

In [ ]:
# The complete walk of Dantzig's rule on KM(3), on the rotatable cube.
A_km, b_km, c_km = klee_minty(3)
res_km = simplex(A_km, b_km, c_km, pivot_rule=dantzig_rule)
S = np.array([1.0, 9.0, 81.0])                 # display rescaling, as above
A_full = np.vstack([A_km, -np.eye(3)]) * S     # the system in rescaled coordinates
b_full = np.concatenate([b_km, np.zeros(3)])
show_lp_3d(A_full, b_full, c=c_km * S, path=res_km["path"] / S,
           show_tight_sets=False,
           title=f"Dantzig's rule on KM(3): {res_km['pivots']} pivots, "
                 f"{len(res_km['path'])} vertices visited")

### Checkpoint C.2 — could the walk have been short?

In the vertex-edge graph of the Klee–Minty cube, how many edges is the optimum away from the starting vertex? (The 3D scrubber above contains the answer, and it holds for every $n$.) Record your prediction; the check cell then verifies it by breadth-first search over the enumerated vertices — using the vertex enumeration of Notebook 1 — for $n = 3, \dots, 6$, and compares with the Hirsch bound $m_{\text{facets}} - n = n$ from the lecture.


In [ ]:
predicted_distance = ...     # C.2: graph distance from start to optimum on KM(n)


In [ ]:
# --- Check for C.2 -----------------------------------------------------------
assert predicted_distance is not Ellipsis, "replace ... by a number first"
print(f"{'n':>3}{'Dantzig pivots':>17}{'graph distance':>17}{'Hirsch bound':>15}")
for n in range(3, 7):
    A, b, c = klee_minty(n)
    r = simplex(A, b, c, pivot_rule=dantzig_rule)
    d = graph_distance(A, b, r["path"][0], r["x"])
    print(f"{n:>3}{r['pivots']:>17}{d:>17}{n:>15}")
    assert predicted_distance == d, \
        (f"the BFS distance is {d} — look again at where the optimum sits "
         "in the 3D scrubber")
print("\nC.2 correct: the optimum is ONE edge away from the start — the face")
print("{x_1 = ... = x_(n-1) = 0} is precisely the segment joining them.")
print("Dantzig's rule walks 2^n - 1 steps around the entire cube to reach a")
print("vertex adjacent to where it began.")


## Where this leads

- The gap on display — $2^n - 1$ pivots against a graph distance of $1$ — separates two questions that the lecture treats in turn: the **diameter** of the polyhedron limits what *any* rule could achieve (Kalai–Kleitman gives the quasi-polynomial bound), while the **pivot rule** determines what is actually achieved.
- The Klee–Minty instance is brittle. Lecture 3 develops the modern explanation of simplex's practical speed — **smoothed analysis** — in which small random perturbations of the data destroy such constructions; Notebook 3 lets you watch this happen.

**Bonus (optional).** The exponential example is tailored to Dantzig's rule. The cell below runs the random-edge rule from Part B on the same instances: it escapes quickly here, although, as the lecture notes, explicit (far more intricate) exponential instances are known for most pivot rules, this one included.


In [ ]:
print(f"{'n':>3}{'Dantzig':>10}{'random edge (mean of 20 runs)':>32}")
for n in range(3, 10):
    A, b, c = klee_minty(n)
    trials = [simplex(A, b, c, pivot_rule=random_edge_rule, seed=s)["pivots"]
              for s in range(20)]
    print(f"{n:>3}{2**n - 1:>10}{np.mean(trials):>32.1f}")
